In [190]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [ ]:
def show (text):
    try:
        Console().print(text)
    except:
        print (text)

In [193]:
openai = OpenAI()

In [194]:
todos = []
completed = []

In [195]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [196]:
get_todo_report()

TypeError: Console.export_html() got an unexpected keyword argument 'inline'

In [ ]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False]* len(descriptions))
    return get_todo_report()

In [ ]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index"
    Console().print(completion_notes)
    return get_todo_report()

In [ ]:
todo, completed = [], []
create_todos(["Buy groceries", "FInish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: FInish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: FInish extra lab\nTodo #3: Eat banana\n'

In [ ]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: FInish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: FInish extra lab\nTodo #3: Eat banana\n'

In [ ]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                "type": "array",
                "items": {"type": "string"},
                "title": "Description"

            }
        },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [ ]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from index 1) and return the full list",
    "parameters": {
        "properties": {
            "index": {
                "description": 'The 1-based index of the todo to mark as complete',
                "title": 'Index',
                "type": 'integer'
            },
            "completion_notes": {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
            }
        },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [ ]:
tools = [
    {'type': 'function', 'function': create_todos_json},
    {'type': 'function', 'function': mark_complete_json}
]

In [ ]:
def handle_tools_calls(tool_calls):
    results =  []
    for  tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        # print (tool)
        # print (arguments)
        result = tool(**arguments) if tool else {}
        # print (result)
        results.append({'role': 'tool', "content": json.dumps(result), 'tool_call_id': tool_call.id})
    return results

In [ ]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages = messages, tools = tools,reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason == 'tool_calls':
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tools_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [ ]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A person is studying  in computer science in his bachelor degree.
How can he be millionaire in 10 years?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [ ]:
todos, completed = [], []
loop(messages)

<function create_todos at 0x7207e8cb9a80>
{'descriptions': ['Interpret the goal and set realistic assumptions (what ‘millionaire’ means; timeframe; starting point).', 'Outline main wealth-building levers for a CS bachelor: income growth, saving/investing rate, equity/ownership, entrepreneurship.', 'Estimate required monthly/annual investment to reach $1M in 10 years under plausible return rates; provide a few scenarios.', 'Create an actionable 10-year roadmap (skills, career moves, side projects, networking, personal finance).', 'List concrete roles/paths (FAANG/quant, startups, freelancing, SaaS) and what to do in years 0–2, 3–5, 6–10.', 'Include risk management, common mistakes, and metrics to track.', 'Present final answer in Rich console markup, concise but practical, without asking questions.']}


Todo #1: Interpret the goal and set realistic assumptions (what ‘millionaire’ means; timeframe; starting point).
Todo #2: Outline main wealth-building levers for a CS bachelor: income growth, saving/investing rate, 
equity/ownership, entrepreneurship.
Todo #3: Estimate required monthly/annual investment to reach $1M in 10 years under plausible return rates; provide
a few scenarios.
Todo #4: Create an actionable 10-year roadmap (skills, career moves, side projects, networking, personal finance).
Todo #5: List concrete roles/paths (FAANG/quant, startups, freelancing, SaaS) and what to do in years 0–2, 3–5, 
6–10.
Todo #6: Include risk management, common mistakes, and metrics to track.
Todo #7: Present final answer in Rich console markup, concise but practical, without asking questions.

Todo #1: Interpret the goal and set realistic assumptions (what ‘millionaire’ means; timeframe; starting point).
Todo #2: Outline main wealth-building levers for a CS bachelor: income growth, saving/investing rate, equity/ownership, entrepreneurship.
Todo #3: Estimate required monthly/annual investment to reach $1M in 10 years under plausible return rates; provide a few scenarios.
Todo #4: Create an actionable 10-year roadmap (skills, career moves, side projects, networking, personal finance).
Todo #5: List concrete roles/paths (FAANG/quant, startups, freelancing, SaaS) and what to do in years 0–2, 3–5, 6–10.
Todo #6: Include risk management, common mistakes, and metrics to track.
Todo #7: Present final answer in Rich console markup, concise but practical, without asking questions.

<function mark_complete at 0x7207e8cb8180>
{'index': 1, 'completion_notes': 'Assumptions used:\n- \x1b[bold]\x1b[bold]Millionaire = net worth \x1b[bold]$\x1b[bold]1,000,000\x1b[bold] (assets \x1b[bold]- deb

Assumptions used:
- Millionaire = net worth $1,000,000 (assets - debts) in 10 years.
- Starting at ~$0 net worth (student), some student debt possible.
- Achievable primarily through high income + high savings + investing and ideally some equity/ownership.
- Use long-run diversified portfolio return range 5-10% nominal (not guaranteed).

Todo #1: Interpret the goal and set realistic assumptions (what ‘millionaire’ means; timeframe; starting point).
Todo #2: Outline main wealth-building levers for a CS bachelor: income growth, saving/investing rate, 
equity/ownership, entrepreneurship.
Todo #3: Estimate required monthly/annual investment to reach $1M in 10 years under plausible return rates; provide
a few scenarios.
Todo #4: Create an actionable 10-year roadmap (skills, career moves, side projects, networking, personal finance).
Todo #5: List concrete roles/paths (FAANG/quant, startups, freelancing, SaaS) and what to do in years 0–2, 3–5, 
6–10.
Todo #6: Include risk management, common mistakes, and metrics to track.
Todo #7: Present final answer in Rich console markup, concise but practical, without asking questions.

...

Todo #1: [green][strike]Interpret the goal and set realistic assumptions (what ‘millionaire’ means; timeframe; starting point).[/strike][/green]
Todo #2: [green][strike]Outline main wealth-building levers for a CS bachelor: income growth, saving/investing rate, equity/ownership, entrepreneurship.[/strike][/green]
Todo #3: [green][strike]Estimate required monthly/annual investment to reach $1M in 10 years under plausible return rates; provide a few scenarios.[/strike][/green]
Todo #4: [green][strike]Create an actionable 10-year roadmap (skills, career moves, side projects, networking, personal finance).[/strike][/green]
Todo #5: [green][strike]List concrete roles/paths (FAANG/quant, startups, freelancing, SaaS) and what to do in years 0–2, 3–5, 6–10.[/strike][/green]
Todo #6: [green][strike]Include risk management, common mistakes, and metrics to track.[/strike][/green]
Todo #7: Present final answer in Rich console markup, concise but practical, without asking questions.

<function mark

Prepared final response in Rich console markup, practical, no questions asked, concise but actionable.

Todo #1: Interpret the goal and set realistic assumptions (what ‘millionaire’ means; timeframe; starting point).
Todo #2: Outline main wealth-building levers for a CS bachelor: income growth, saving/investing rate, 
equity/ownership, entrepreneurship.
Todo #3: Estimate required monthly/annual investment to reach $1M in 10 years under plausible return rates; provide
a few scenarios.
Todo #4: Create an actionable 10-year roadmap (skills, career moves, side projects, networking, personal finance).
Todo #5: List concrete roles/paths (FAANG/quant, startups, freelancing, SaaS) and what to do in years 0–2, 3–5, 
6–10.
Todo #6: Include risk management, common mistakes, and metrics to track.
Todo #7: Present final answer in Rich console markup, concise but practical, without asking questions.

Todo #1: [green][strike]Interpret the goal and set realistic assumptions (what ‘millionaire’ means; timeframe; starting point).[/strike][/green]
Todo #2: [green][strike]Outline main wealth-building levers for a CS bachelor: income growth, saving/investing rate, equity/ownership, entrepreneurship.[/strike][/green]
Todo #3: [green][strike]Estimate required monthly/annual investment to reach $1M in 10 years under plausible return rates; provide a few scenarios.[/strike][/green]
Todo #4: [green][strike]Create an actionable 10-year roadmap (skills, career moves, side projects, networking, personal finance).[/strike][/green]
Todo #5: [green][strike]List concrete roles/paths (FAANG/quant, startups, freelancing, SaaS) and what to do in years 0–2, 3–5, 6–10.[/strike][/green]
Todo #6: [green][strike]Include risk management, common mistakes, and metrics to track.[/strike][/green]
Todo #7: [green][strike]Present final answer in Rich console markup, concise but practical, without asking questions.[

Goal (realistic definition)
A “millionaire in 10 years” usually means net worth ≥ $1,000,000 (investments + cash + assets − debts) within 10 
years after starting your career.

What it typically requires (math)
If you start near $0 and invest consistently for 10 years:
- At ~5% annual return: need about $79k/year invested (~$6.6k/month)
- At ~7% annual return: need about $72k/year invested (~$6.0k/month)
- At ~10% annual return: need about $63k/year invested (~$5.2k/month)

So the fastest realistic routes are:
1) Very high income + high savings rate, and/or  
2) Equity/ownership outcome (RSUs, startup equity, your own product/business)

The 4 levers to pull (in order)
1. Increase income fast (skills + targeting high-paying markets + negotiation)
2. Keep expenses controlled (avoid lifestyle inflation)
3. Invest automatically (broad index funds/ETFs; diversify)
4. Add ownership (equity or a scalable side business)

A practical 10-year roadmap for a CS bachelor

Years 0–2: Build a “high-comp” profile
- Pick 1–2 strong tracks:
  - Backend + distributed systems + cloud (high demand)
  - Security (high leverage)
  - ML engineering (if you can build + ship)
  - Quant/low-latency C++ (harder, very high pay)
- Do internships (even unpaid is better than none early, but aim for paid quickly).
- Build a portfolio that proves you can ship:
  - 2–3 real projects (deployed, monitored, documented)
  - one “business-like” project (payments/subscriptions or B2B tool)
- Interview prep seriously (DSA + system design). High-paying offers are often gated by interviews.

Years 2–5: Maximize compensation + savings rate
- Target roles with strong total compensation:
  - Big tech/product companies (salary + bonus + RSUs)
  - Quant/HFT/fintech (high cash comp)
  - High-growth startups (salary + equity; more risk)
- Strategy:
  - Negotiate every offer; switch companies if growth stalls.
  - Aim for rapid promotion or hop every ~18–30 months if it increases total comp.
- Money system:
  - Build 3–6 months emergency fund.
  - Eliminate high-interest debt.
  - Automate investing monthly (broad diversified funds).
  - Maintain a high savings rate (many millionaires do 40–70% early when income rises).

Years 5–10: Add ownership for the “jump”
Pick at least one ownership path (alongside your job until it’s real):
- RSU-heavy track: stay in big tech, grow level, keep expenses stable, invest RSU proceeds (don’t let one stock 
dominate).
- Startup equity track: join early with strong team/traction; negotiate equity; understand vesting and dilution 
(risk is high, but upside can be large).
- SaaS/app track: build a small product that earns recurring revenue:
  - solve one painful niche problem
  - charge early
  - distribution matters (content, partnerships, sales outreach)
- Consulting → agency: freelance to $10k–$30k/month, then hire and productize.

Concrete targets that make $1M plausible
- Get to $150k–$300k+ total compensation within a few years (market-dependent).
- Keep annual spending relatively flat while income grows.
- Invest aggressively and consistently (ideally $60k–$100k/year if aiming for $1M in 10 years without a big equity 
win).
- Build at least one “equity/ownership bet” that could add an extra $200k–$800k+ upside.

Risk management (don’t blow the plan)
- Avoid lifestyle inflation (cars, luxury rent) when pay rises.
- Don’t rely on one stock/one crypto bet for the whole goal.
- Keep skills current (cloud, systems, security, data) so income doesn’t drop.
- Insure against disasters (health/disability as applicable) and keep an emergency fund.

Simple metrics to track monthly
- Savings rate %
- Invested assets total (net worth trend)
- Income and total comp
- Time spent on ownership project + its revenue/users

If you execute the “high income + high savings + consistent investing” plan, millionaire in ~10 years is possible; 
if you add a successful equity/ownership outcome, it becomes much more likely (and faster).